# Sucking Problem

This is a testcase for evaporation.  
Some liquid evaporated at a planar fluid vapor interface.  
The evaporation is driven by an initially superheated liquid.  
Initially $t = 0$ there is no vapor in the system.  
The simulation is started from an analytical solution at $t_0 > 0$.  
We can then compare the simulated interface position in time to the analytically predicted.  
See `10.1016/j.ijheatmasstransfer.2021.121233`  

Note the following:
* A deviation from the source is the considerably larger surface tension, to deal with capillary timestep restrictions.
* If we initialize with zero velocity fields the interface movement is off in the first timestep. There are two workarounds: initialize with the analytical velocity (done now) or use a very small first timestep.
* Paraview seems to not read time data correcty. Thus time annotation is missing in videos.

In [1]:
#r "..\..\binaries\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [2]:
string proj = "XNSFE-SuckingProblem-v1";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

Project name is set to 'XNSFE-SuckingProblem-v1'.
Opening existing database '\\dc1\userspace\rieckmann\cluster\XNSFE-SuckingProblem-v1'.


In [3]:
var sessions = BoSSSshell.WorkflowMgm.DefaultDatabase.Sessions;
sessions

#0: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P4_T3_LieSplitting_SDIRK_33	11/29/2022 19:12:49	77d93215...
#1: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P4_T3_StrangSplitting_SDIRK_33	11/29/2022 19:12:34	4e66ac19...
#2: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P4_T3_LieSplitting_SDIRK_22	11/29/2022 19:12:21	5e93b093...
#3: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P4_T3_StrangSplitting_SDIRK_22	11/29/2022 19:12:06	f0732577...
#4: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P4_T2_LieSplitting_SDIRK_33	11/29/2022 19:11:00	1eaa4f84...
#5: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P4_T2_StrangSplitting_SDIRK_33	11/29/2022 19:10:48	dc8a4bbd...
#6: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P4_T2_LieSplitting_SDIRK_22	11/29/2022 19:10:35	d49f0339...
#7: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P4_T2_StrangSplitting_SDIRK_22	11/29/2022 19:10:19	ec32e43a...
#8: XNSFE-SuckingPro

## Grid and Boundary/Initial Value configuration

We make use of the following connections ($A$ - Liquid, $B$ - Vapor),
as the problem is quasi-1D, only the x-directional velocity components are considerd :  
Known values from vapor side:    
* $T_B(x,t) = T_{sat}$
* $u_B(x,t) = 0$
Interface velocity :  
* $x_I(t) = 2 * \xi * \sqrt{\alpha_B * t}$
* $\dot{x}_I(t) = \xi * \sqrt{\frac{\alpha_B}{t}}$
* $\xi = 0.7677$
* $\alpha_{A/B} = \frac{k_{A/B}}{\rho_{A/B}c_{A/B}}$  
Fluxes at the interface : 
* $\dot{m} = -\rho_B(u_B - \dot{x}) = \rho_B \dot{x}$
* $h_V\dot{m} = k_A T_{x,A}$  
Therefore :  
* $T_{x,A} = \frac{\rho_B h_V}{k_A} \xi \sqrt{\alpha_B t}$  
this temperature gradient at the interface is used for extrapolating the temperature profile in phase $A$ for setting the initial conditions.

Rescale temperature, for some reason better for the solver:  
$\Theta = \frac{T - T_{s}}{T_{s} - T_{Wall}}$  
$\Delta T = {T_{s} - T_{Wall}}$  

$ \tilde{c} = \Delta T * c$  
$ \tilde{k} = \Delta T * k$  

In [4]:
static class Constants{
    // careful order of declaration matters!!!
    public static double L = 0.01;
    public static double Xi = 0.7677; // see Matlab Sheet SuckingProblem.m, note the different formulas for interface position in SuckingProblem.m and the reference work. In the following we use the constants stated in the reference.

    // material parameters
    public static double rho_A = 958.4;
    public static double rho_B = 0.597;

    public static double mu_A = 2.8e-4;
    public static double mu_B = 1.26e-5;

    public static double Sigma = 0.059 / 1000.0; // very small surface tension so that capillary timestep is bigger...

    public static double c_A = 4216.0 * 5.0;
    public static double c_B = 2030.0 * 5.0;

    public static double k_A = 0.679 * 5.0;
    public static double k_B = 0.025 * 5.0;

    public static double hVap  = 2.258e6;
    public static double T_sat = 0.0;//373.15;
    public static double T_wall = T_sat + 1.0;//T_sat + 5.0;

    public static double alpha_A = k_A / (c_A*rho_A);
    public static double alpha_B = k_B / (c_B*rho_B);    
    public static double C = k_A / (rho_B*hVap);
    public static double eps = 1.0-rho_B/rho_A;

    public static double x0 = 0.00221;
    public static double t0 = Math.Pow(0.5 * x0 / Xi, 2) / alpha_B; // start time, to avoid singular massflux at t=0
    public static double tE = 0.6 - t0; // Endtime
    // capillary timestep , for finest res + highest degree, use this, for comparability?!, Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) => 0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [5]:
static class GridFactory{
    public static GridCommons Grid(int res, IDatabaseInfo myDb){
        double h = Constants.L/res;
        double[] Xnodes = GenericBlas.Linspace(0, Constants.L, res + 1);
        double[] Ynodes = GenericBlas.Linspace(0, h, 2); // quasi 1D always use one cell
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "freeslip_ZeroGradient");
        grd.EdgeTagNames.Add(2, "wall_ConstantTemperature");
        grd.EdgeTagNames.Add(3, "pressure_outlet_ConstantTemperature");
        
        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 1;
            if(Math.Abs(X[0]) < 1e-6)
                return 2;
            if(Math.Abs(X[0] -  Constants.L) < 1e-6)
                return 3;
            return et;
        });

        myDb.Controller.DBDriver.SaveGrid(grd, myDb);

        return grd;
    }
}

In [6]:
public static class BoundaryAndInitialValueFactory { 

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfacePos(){");
            stw.WriteLine("         return (X,t) => 2 * " + Constants.Xi + " * Math.Sqrt(" + Constants.alpha_B + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfaceVel(){");
            stw.WriteLine("         return (X,t) => " + Constants.Xi + " * " + Constants.alpha_B + " / Math.Sqrt(" + Constants.alpha_B + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){");
            stw.WriteLine("         return InterfacePos()(X,t) - X[0];");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureB(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_sat + "; // always saturation temp.");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double TemperatureABoundary(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double TemperatureBBoundary(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_sat + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double VelocityA(double[] X, double t){");
            stw.WriteLine("         return " + Constants.eps + " * InterfaceVel()(X,t);");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityB(double[] X, double t){");
            stw.WriteLine("         return 0.0;");
            stw.WriteLine("     }");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }
   
    static public Formula Get_VA() {
        return new Formula("BoundaryAndInitialValues.VelocityA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempB() {
        return new Formula("BoundaryAndInitialValues.TemperatureB", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempB_Boundary() {
        return new Formula("BoundaryAndInitialValues.TemperatureBBoundary", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempA_Boundary() {
        return new Formula("BoundaryAndInitialValues.TemperatureABoundary", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode:GetPrefixCode());
    }
}

In [7]:
// load reference solution data, see Matlab Sheet SuckingProblem.m, to initialize the temperature profile. This is also a similarity solution, used as reference later on
string path = @".\SuckingProblemRef.csv";
string[] lines = File.ReadAllLines(path);
double[] eta = new double[lines.Length];
double[][] valueDat = new double[2][];
for(int j = 0; j < 2; j++)
    valueDat[j] = new double[lines.Length];

for (int i = 0; i < lines.Length; i++) {
    eta[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[0]);            
    for (int j = 0; j < 2; j++) {
        valueDat[j][i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[j+1]);
        if(j == 0) valueDat[j][i] = (valueDat[j][i] - 373.15) / 5.0;
    }
} 

In [8]:
// use loaded data, to construct a spline as the initial condition
double x0 = Constants.x0;
double[] x = new double[lines.Length+2];
double[] y = new double[lines.Length+2];
x[0] = 0.0;
y[0] = Constants.T_sat-x0 * Constants.Xi / Constants.C * Math.Sqrt(Constants.alpha_B / Constants.t0) ; // linear extrapolation node, using slope of temperature at interface
for(int i = 1; i < lines.Length + 1; i++){
    y[i] = valueDat[0][i-1];
    double xi = eta[i-1]* Math.Sqrt(2 * Constants.alpha_A * Constants.t0);
    x[i] = x0 + xi;
}
x[lines.Length+1] = Constants.L;
y[lines.Length+1] = Constants.T_wall;
Spline1D TempAInitial = new Spline1D(x,y,0);

In [9]:
Gnuplot plt = new Gnuplot();
plt.PlotFunction(delegate (MultidimensionalArray X, MultidimensionalArray Y) {TempAInitial.EvaluateV(X,0.0,Y);}, BoSSS.Foundation.Grid.Classic.Grid1D.LineGrid(GenericBlas.Linspace(Constants.x0,0.008,1000)).GridData);
plt.PlotNow()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.2 
 
 
 
 
 0.4 
 
 
 
 
 0.6 
 
 
 
 
 0.8 
 
 
 
 
 1 
 
 
 
 
 1.2 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 0.008 
 
 
 
 
 0.009 
 
 
 
 
 
 
 
 
 BoSSS.Foundation.ScalarFunction 
 
 
 BoSSS.Foundation.ScalarFunction 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M716.2,34.7 L758.4,34.7 M75.5,564.0 L76.1,552.5 L76.7,540.9 L77.3,529.4 L77.9,517.9 L78.5,506.5
 L79.1,495.2 L79.7,483.9 L80.3,472.7 L80.9,461.5 L81.5,450.5 L82.1,439.6 L82.7,428.9 L83.3,418.2
 L83.9,407.7 L84.5,397.3 L85.1,387.1 L85.7,377.1 L86.3,367.2 L86.9,357.5 L87.5,348.0 L88.1,338.7
 L88.7,329.5 L89.3,320.6 L89.9,311.9 L90.5,303.4 L91.1,295.1 L91.7,287.0 L92.3,279.1 L92.8,271.4
 L93.4,264.0 L94.0,256.8 L94.6,249.8 L95.2,243.0 L95.8,236.4 L96.4,230.1 L97.0,224.0 L97.6,218.1
 L98.2,212.4 L98.8,206.9 L99.4,201.7 L100.0,196.6 L100.6,191.7 L101.2,187.1 L101.8,182.6 L102.4,178.3
 L103.0,174.2 L103.6,170.3 L104.2,166.6 L104.8,163.1 L105.4,159.7 L106.0,156.5 L106.6,153.4 L107.2,150.5
 L107.8,147.8 L108.4,145.2 L109.0,142.8 L109.6,140.4 L110.2,138.3 L110.8,136.2 L111.4,134.3 L112.0,132.4
 L112.6,130.7 L113.1,129.1 L113.7,127.6 L114.3,126.1 L114.9,124.8 L115.5,123.5 L116.1,122.3 L116.7,121.2
 L117.3,120.2 L117.9,119.2 L118.5,118.3 L119.1,117.5 L119.7,116.7 L120.3,116.0 L120.9,115.4 L121.5,114.7
 L122.1,114.2 L122.7,113.6 L123.3,113.1 L123.9,112.7 L124.5,112.3 L125.1,111.9 L125.7,111.5 L126.3,111.2
 L126.9,110.9 L127.5,110.6 L128.1,110.4 L128.7,110.2 L129.3,109.9 L129.9,109.8 L130.5,109.6 L131.1,109.4
 L131.7,109.3 L132.3,109.1 L132.8,109.0 L133.4,108.9 L134.0,108.8 L134.6,108.7 L135.2,108.6 L135.8,108.6
 L136.4,108.5 L137.0,108.4 L137.6,108.4 L138.2,108.3 L138.8,108.3 L139.4,108.2 L140.0,108.2 L140.6,108.2
 L141.2,108.1 L141.8,108.1 L142.4,108.1 L143.0,108.1 L143.6,108.1 L144.2,108.0 L144.8,108.0 L145.4,108.0
 L146.0,108.0 L146.6,108.0 L147.2,108.0 L147.8,108.0 L148.4,108.0 L149.0,108.0 L149.6,108.0 L150.2,108.0
 L150.8,107.9 L151.4,107.9 L152.0,107.9 L152.6,107.9 L153.1,107.9 L153.7,107.9 L154.3,107.9 L154.9,107.9
 L155.5,107.9 L156.1,107.9 L156.7,107.9 L157.3,107.9 L157.9,107.9 L158.5,107.9 L159.1,107.9 L159.7,107.9
 L160.3,107.9 L160.9,107.9 L161.5,107.9 L162.1,107.9 L162.7,107.9 L163.3,107.9 L163.9,107.9 L164.5,107.9
 L165.1,107.9 L165.7,107.9 L166.3,107.9 L166.9,107.9 L167.5,107.9 L168.1,107.9 L168.7,107.9 L169.3,107.9
 L169.9,107.9 L170.5,107.9 L171.1,107.9 L171.7,107.9 L172.3,107.9 L172.9,107.9 L173.4,107.9 L174.0,107.9
 L174.6,107.9 L175.2,107.9 L175.8,107.9 L176.4,107.9 L177.0,107.9 L177.6,107.9 L178.2,107.9 L178.8,107.9
 L179.4,107.9 L180.0,107.9 L180.6,107.9 L181.2,107.9 L181.8,107.9 L182.4,107.9 L183.0,107.9 L183.6,107.9
 L184.2,107.9 L184.8,107.9 L185.4,107.9 L186.0,107.9 L186.6,107.9 L187.2,107.9 L187.8,107.9 L188.4,107.9
 L189.0,107.9 L189.6,107.9 L190.2,107.9 L190.8,107.9 L191.4,107.9 L192.0,107.9 L192.6,107.9 L193.2,107.9
 L193.7,107.9 L194.3,107.9 L194.9,107.9 L195.5,107.9 L196.1,107.9 L196.7,107.9 L197.3,107.9 L197.9,107.9
 L198.5,107.9 L199.1,107.9 L199.7,107.9 L200.3,107.9 L200.9,107.9 L201.5,107.9 L202.1,107.9 L202.7,107.9
 L203.3,107.9 L203.9,107.9 L204.5,107.9 L205.1,107.9 L205.7,107.9 L206.3,107.9 L206.9,107.9 L207.5,107.9
 L208.1,107.9 L208.7,107.9 L209.3,107.9 L209.9,107.9 L210.5,107.9 L211.1,107.9 L211.7,107.9 L212.3,107.9
 L212.9,107.9 L213.5,107.9 L214.0,107.9 L214.6,107.9 L215.2,107.9 L215.8,107.9 L216.4,107.9 L217.0,107.9
 L217.6,107.9 L218.2,107.9 L218.8,107.9 L219.4,107.9 L220.0,107.9 L220.6,107.9 L221.2,107.9 L221.8,107.9
 L222.4,107.9 L223.0,107.9 L223.6,107.9 L224.2,107.9 L224.8,107.9 L225.4,107.9 L226.0,107.9 L226.6,107.9
 L227.2,107.9 L227.8,107.9 L228.4,107.9 L229.0,107.9 L229.6,107.9 L2

## Begin Postprocessing

In [10]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination && Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "H");
sessions = sessions.OrderBy(s => s.KeysAndQueries["Grid:hMax"]).ToList();
sessions

#0: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P4	11/14/2022 13:20:50	1ea7f318...
#1: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P2	11/14/2022 13:20:31	609eb176...
#2: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P3	11/14/2022 13:20:41	9d51c326...
#3: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P1	11/14/2022 13:20:21	57ebe3d7...
#4: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H32_AMR0_P4	11/14/2022 13:20:12	82541579...
#5: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H32_AMR0_P3	11/14/2022 13:20:02	760d3d43...
#6: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H32_AMR0_P1	11/14/2022 13:19:43	dcd82dde...
#7: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H32_AMR0_P2	11/14/2022 13:19:53	834c1c20...
#8: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H16_AMR0_P3	11/14/2022 13:19:25	41975e70...
#9: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H16_AMR0_P4	11/14/2022 13:19:35	93b4ca77...
#10: XNSFE-SuckingProblem-v1	XNSFE-Sucki

Lets sort the sessions by degree

In [11]:
SortedDictionary<int, List<ISessionInfo>> groupedSessions = new SortedDictionary<int, List<ISessionInfo>>(sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"])).ToDictionary(g => g.Key, g => g.ToList()));

Load Log Data

In [12]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));

Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet


### Growthrate


### Interface Position

Evaluate the analyical solution for the interface position

In [13]:
public static void subsample(Plot2Ddata plt, int n){
    foreach(var group in plt.dataGroups){
        var X = group.Abscissas;
        var Y = group.Values;
        int N = X.Count();
        List<double> newX = new List<double>();
        List<double> newY = new List<double>();
        
        newX.Add(X[0]);
        newY.Add(Y[0]);
        for(int i = 1; i < N - 1; i++){
            if((i % n) == 0){
                newX.Add(X[i]);
                newY.Add(Y[i]);
            }
        }
        newX.Add(X[N-1]);
        newY.Add(Y[N-1]);
        
        group.Abscissas = newX.ToArray();
        group.Values = newY.ToArray();
    }
}

In [14]:
data.ForEach(dat => dat.Value.ForEach(pp => subsample(pp, 50)));

In [15]:
double[] times = GenericBlas.Linspace(0.0, Constants.tE + Constants.t0, 1000);
double[] yValues = new double[times.Length];
var PhiFuncF = BoundaryAndInitialValueFactory.Get_Phi();
Func<double, double> PhiFunc = t => {
    return PhiFuncF.Evaluate(new double[2], t);
};
for(int i = 0; i < times.Length; i++){
    yValues[i] = PhiFunc(times[i] - Constants.t0);
}
Plot2Ddata RefPos = new Plot2Ddata(times, yValues, "analytical Interface Position");
RefPos.dataGroups.First().Format = new PlotFormat("r-");

In [17]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    var plot = dat.ElementAt(0);
    plot.ModFormat();
     foreach(var grp in plot.dataGroups){
        grp.Name = grp.Name.Split('-').ElementAt(3).Replace("H", "n=");
        grp.Abscissas = grp.Abscissas.Select(x => x + Constants.t0).ToArray();
    };
    plot = plot.Merge(RefPos);
    plot.LegendAlignment = new string[] {"i", "t", "l"}; 
    plot.Ylabel = "Interface position [m]";
    plot.Xlabel = "Time [s]";
    plot.LabelFont = 16;
    plot.LabelTitleFont = 16;
    plot.Title = "p=" + p;
    plot.TitleFont = 24;

    var gp = plot.ToGnuplot();
    gp.WriteScriptAndData("./Figures/gnuplot/data/Suckingproblem_InterfacePos_Zoom_p"+p); // change a few settings by hand, see readme!
    var p1 = plot.PlotNow(); 
    
    plot.ShowLegend = false;
    plot.XrangeMin = Constants.tE ;
    plot.XrangeMax = Constants.tE+ Constants.t0;

    plot.YrangeMin = 0.99 * PhiFunc(plot.XrangeMin.Value);
    plot.YrangeMax = 1.01 * PhiFunc(plot.XrangeMax.Value);
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    // display(pp);
}

Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp5

In [18]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));
data.ForEach(dat => dat.Value.ForEach(pp => subsample(pp, 50)));
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    var plot = dat.ElementAt(0);
    plot.ModFormat();
     foreach(var grp in plot.dataGroups){
        grp.Name = grp.Name.Split('-').ElementAt(3).Replace("H", "n=");
        grp.Abscissas = grp.Abscissas.Select(x => x + Constants.t0).ToArray();
    };
    plot = plot.Merge(RefPos);
    plot.LegendAlignment = new string[] {"i", "t", "l"}; 
    plot.Ylabel = "Interface position [m]";
    plot.Xlabel = "Time [s]";
    plot.LabelFont = 16;
    plot.LabelTitleFont = 16;
    plot.Title = "p=" + p;
    plot.TitleFont = 24;

    var gp = plot.ToGnuplot();
    gp.WriteScriptAndData("./Figures/gnuplot/data/Suckingproblem_InterfacePos_p"+p); // change a few settings by hand, see readme!
    var p1 = plot.PlotNow(); 
    
    plot.ShowLegend = false;
    plot.XrangeMin = Constants.tE ;
    plot.XrangeMax = Constants.tE+ Constants.t0;

    plot.YrangeMin = 0.99 * PhiFunc(plot.XrangeMin.Value);
    plot.YrangeMax = 1.01 * PhiFunc(plot.XrangeMax.Value);
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp5

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Interface position [m] 
 
 
 
 
 Time [s] 
 
 
 
 
 p=1 
 
 
 n=64 
 
 
 n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=32 
 
 
 n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=16 
 
 
 n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=8 
 
 
 n=8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 analytical Interface Position 
 
 
 analytical Interface Position 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M107.9,207.1 L161.3,207.1 M88.5,542.4 L89.1,529.5 L89.6,524.2 L90.2,520.1 L90.8,516.6 L91.3,513.6
 L91.9,510.8 L92.5,508.3 L93.0,506.0 L93.6,503.8 L94.2,501.7 L94.7,499.7 L95.3,497.8 L95.9,495.9
 L96.4,494.2 L97.0,492.5 L97.6,490.9 L98.2,489.3 L98.7,487.7 L99.3,486.2 L99.9,484.8 L100.4,483.4
 L101.0,482.0 L101.6,480.6 L102.1,479.3 L102.7,478.0 L103.3,476.7 L103.8,475.5 L104.4,474.2 L105.0,473.0
 L105.5,471.8 L106.1,470.7 L106.7,469.5 L107.2,468.4 L107.8,467.3 L108.4,466.2 L108.9,465.1 L109.5,464.0
 L110.1,463.0 L110.6,461.9 L111.2,460.9 L111.8,459.9 L112.3,458.9 L112.9,457.9 L113.5,456.9 L114.0,456.0
 L114.6,455.0 L115.2,454.1 L115.7,453.1 L116.3,452.2 L116.9,451.3 L117.5,450.4 L118.0,449.5 L118.6,448.6
 L119.2,447.7 L119.7,446.9 L120.3,446.0 L120.9,445.1 L121.4,444.3 L122.0,443.4 L122.6,442.6 L123.1,441.8
 L123.7,441.0 L124.3,440.1 L124.8,439.3 L125.4,438.5 L126.0,437.7 L126.5,436.9 L127.1,436.2 L127.7,435.4
 L128.2,434.6 L128.8,433.8 L129.4,433.1 L129.9,432.3 L130.5,431.6 L131.1,430.8 L131.6,430.1 L132.2,429.4
 L132.8,428.6 L133.3,427.9 L133.9,427.2 L134.5,426.5 L135.0,425.7 L135.6,425.0 L136.2,424.3 L136.8,423.6
 L137.3,422.9 L137.9,422.2 L138.5,421.5 L139.0,420.9 L139.6,420.2 L140.2,419.5 L140.7,418.8 L141.3,418.2
 L141.9,417.5 L142.4,416.8 L143.0,416.2 L143.6,415.5 L144.1,414.9 L144.7,414.2 L145.3,413.6 L145.8,412.9
 L146.4,412.3 L147.0,411.6 L147.5,411.0 L148.1,410.4 L148.7,409.8 L149.2,409.1 L149.8,408.5 L150.4,407.9
 L150.9,407.3 L151.5,406.7 L152.1,406.1 L152.6,405.4 L153.2,404.8 L153.8,404.2 L154.3,403.6 L154.9,403.0
 L155.5,402.5 L156.1,401.9 L156.6,401.3 L157.2,400.7 L157.8,400.1 L158.3,399.5 L158.9,398.9 L159.5,398.4
 L160.0,397.8 L160.6,397.2 L161.2,396.6 L161.7,396.1 L162.3,395.5 L162.9,394.9 L163.4,394.4 L164.0,393.8
 L164.6,393.3 L165.1,392.7 L165.7,392.2 L166.3,391.6 L166.8,391.1 L167.4,390.5 L168.0,390.0 L168.5,389.4
 L169.1,388.9 L169.7,388.3 L170.2,387.8 L170.8,387.3 L171.4,386.7 L171.9,386.2 L172.5,385.7 L173.1,385.1
 L173.6,384.6 L174.2,384.1 L174.8,383.6 L175.4,383.0 L175.9,382.5 L176.5,382.0 L177.1,381.5 L177.6,381.0
 L178.2,380.5 L178.8,379.9 L179.3,379.4 L179.9,378.9 L180.5,378.4 L181.0,377.9 L181.6,377.4 L182.2,376.9
 L182.7,376.4 L183.3,375.9 L183.9,375.4 L184.4,374.9 L185.0,374.4 L185.6,373.9 L186.1,373.4 L186.7,372.9
 L187.3,372.5 L187.8,372.0 L188.4,371.5 L189.0,371.0 L189.5,370.5 L190.1,370.0 L190.7,369.6 L191.2,369.1
 L191.8,368.6 L192.4,368.1 L192.9,367.6 L193.5,367.2 L194.1,366.7 L194.7,366.2 L195.2,365.8 L195.8,365.3
 L196.4,364.8 L196.9,364.4 L197.5,363.9 L198.1,363.4 L198.6,363.0 L199.2,362.5 L199.8,362.0 L200.3,361.6
 L200.9,361.1 L201.5,360.7 L202.0,360.2 L202.6,359.7 L203.2,359.3 L203.7,358.8 L204.3,358.4 L204.9,357.9
 L205.4,357.5 L206.0,357.0 L206.6,356.6 L207.1,356.1 L207.7,355.7 L208.3,355.3 L208.8,354.8 L209.4,354.4
 L210.0,353.9 L210.5,353.5 L211.1,353.1 L211.7,352.6 L212.2,352.2 L212.8,351.7 L213.4,351.3 L214.

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Interface position [m] 
 
 
 
 
 Time [s] 
 
 
 
 
 p=2 
 
 
 n=64 
 
 
 n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=32 
 
 
 n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=16 
 
 
 n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=8 
 
 
 n=8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 analytical Interface Position 
 
 
 analytical Interface Position 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M107.9,207.1 L161.3,207.1 M88.5,542.4 L89.1,529.5 L89.6,524.2 L90.2,520.1 L90.8,516.6 L91.3,513.6
 L91.9,510.8 L92.5,508.3 L93.0,506.0 L93.6,503.8 L94.2,501.7 L94.7,499.7 L95.3,497.8 L95.9,495.9
 L96.4,494.2 L97.0,492.5 L97.6,490.9 L98.2,489.3 L98.7,487.7 L99.3,486.2 L99.9,484.8 L100.4,483.4
 L101.0,482.0 L101.6,480.6 L102.1,479.3 L102.7,478.0 L103.3,476.7 L103.8,475.5 L104.4,474.2 L105.0,473.0
 L105.5,471.8 L106.1,470.7 L106.7,469.5 L107.2,468.4 L107.8,467.3 L108.4,466.2 L108.9,465.1 L109.5,464.0
 L110.1,463.0 L110.6,461.9 L111.2,460.9 L111.8,459.9 L112.3,458.9 L112.9,457.9 L113.5,456.9 L114.0,456.0
 L114.6,455.0 L115.2,454.1 L115.7,453.1 L116.3,452.2 L116.9,451.3 L117.5,450.4 L118.0,449.5 L118.6,448.6
 L119.2,447.7 L119.7,446.9 L120.3,446.0 L120.9,445.1 L121.4,444.3 L122.0,443.4 L122.6,442.6 L123.1,441.8
 L123.7,441.0 L124.3,440.1 L124.8,439.3 L125.4,438.5 L126.0,437.7 L126.5,436.9 L127.1,436.2 L127.7,435.4
 L128.2,434.6 L128.8,433.8 L129.4,433.1 L129.9,432.3 L130.5,431.6 L131.1,430.8 L131.6,430.1 L132.2,429.4
 L132.8,428.6 L133.3,427.9 L133.9,427.2 L134.5,426.5 L135.0,425.7 L135.6,425.0 L136.2,424.3 L136.8,423.6
 L137.3,422.9 L137.9,422.2 L138.5,421.5 L139.0,420.9 L139.6,420.2 L140.2,419.5 L140.7,418.8 L141.3,418.2
 L141.9,417.5 L142.4,416.8 L143.0,416.2 L143.6,415.5 L144.1,414.9 L144.7,414.2 L145.3,413.6 L145.8,412.9
 L146.4,412.3 L147.0,411.6 L147.5,411.0 L148.1,410.4 L148.7,409.8 L149.2,409.1 L149.8,408.5 L150.4,407.9
 L150.9,407.3 L151.5,406.7 L152.1,406.1 L152.6,405.4 L153.2,404.8 L153.8,404.2 L154.3,403.6 L154.9,403.0
 L155.5,402.5 L156.1,401.9 L156.6,401.3 L157.2,400.7 L157.8,400.1 L158.3,399.5 L158.9,398.9 L159.5,398.4
 L160.0,397.8 L160.6,397.2 L161.2,396.6 L161.7,396.1 L162.3,395.5 L162.9,394.9 L163.4,394.4 L164.0,393.8
 L164.6,393.3 L165.1,392.7 L165.7,392.2 L166.3,391.6 L166.8,391.1 L167.4,390.5 L168.0,390.0 L168.5,389.4
 L169.1,388.9 L169.7,388.3 L170.2,387.8 L170.8,387.3 L171.4,386.7 L171.9,386.2 L172.5,385.7 L173.1,385.1
 L173.6,384.6 L174.2,384.1 L174.8,383.6 L175.4,383.0 L175.9,382.5 L176.5,382.0 L177.1,381.5 L177.6,381.0
 L178.2,380.5 L178.8,379.9 L179.3,379.4 L179.9,378.9 L180.5,378.4 L181.0,377.9 L181.6,377.4 L182.2,376.9
 L182.7,376.4 L183.3,375.9 L183.9,375.4 L184.4,374.9 L185.0,374.4 L185.6,373.9 L186.1,373.4 L186.7,372.9
 L187.3,372.5 L187.8,372.0 L188.4,371.5 L189.0,371.0 L189.5,370.5 L190.1,370.0 L190.7,369.6 L191.2,369.1
 L191.8,368.6 L192.4,368.1 L192.9,367.6 L193.5,367.2 L194.1,366.7 L194.7,366.2 L195.2,365.8 L195.8,365.3
 L196.4,364.8 L196.9,364.4 L197.5,363.9 L198.1,363.4 L198.6,363.0 L199.2,362.5 L199.8,362.0 L200.3,361.6
 L200.9,361.1 L201.5,360.7 L202.0,360.2 L202.6,359.7 L203.2,359.3 L203.7,358.8 L204.3,358.4 L204.9,357.9
 L205.4,357.5 L206.0,357.0 L206.6,356.6 L207.1,356.1 L207.7,355.7 L208.3,355.3 L208.8,354.8 L209.4,354.4
 L210.0,353.9 L210.5,353.5 L211.1,353.1 L211.7,352.6 L212.2,352.2 L212.8,351.7 L213.4,351.3 L214.

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Interface position [m] 
 
 
 
 
 Time [s] 
 
 
 
 
 p=3 
 
 
 n=64 
 
 
 n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=32 
 
 
 n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=16 
 
 
 n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=8 
 
 
 n=8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 analytical Interface Position 
 
 
 analytical Interface Position 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M107.9,207.1 L161.3,207.1 M88.5,542.4 L89.1,529.5 L89.6,524.2 L90.2,520.1 L90.8,516.6 L91.3,513.6
 L91.9,510.8 L92.5,508.3 L93.0,506.0 L93.6,503.8 L94.2,501.7 L94.7,499.7 L95.3,497.8 L95.9,495.9
 L96.4,494.2 L97.0,492.5 L97.6,490.9 L98.2,489.3 L98.7,487.7 L99.3,486.2 L99.9,484.8 L100.4,483.4
 L101.0,482.0 L101.6,480.6 L102.1,479.3 L102.7,478.0 L103.3,476.7 L103.8,475.5 L104.4,474.2 L105.0,473.0
 L105.5,471.8 L106.1,470.7 L106.7,469.5 L107.2,468.4 L107.8,467.3 L108.4,466.2 L108.9,465.1 L109.5,464.0
 L110.1,463.0 L110.6,461.9 L111.2,460.9 L111.8,459.9 L112.3,458.9 L112.9,457.9 L113.5,456.9 L114.0,456.0
 L114.6,455.0 L115.2,454.1 L115.7,453.1 L116.3,452.2 L116.9,451.3 L117.5,450.4 L118.0,449.5 L118.6,448.6
 L119.2,447.7 L119.7,446.9 L120.3,446.0 L120.9,445.1 L121.4,444.3 L122.0,443.4 L122.6,442.6 L123.1,441.8
 L123.7,441.0 L124.3,440.1 L124.8,439.3 L125.4,438.5 L126.0,437.7 L126.5,436.9 L127.1,436.2 L127.7,435.4
 L128.2,434.6 L128.8,433.8 L129.4,433.1 L129.9,432.3 L130.5,431.6 L131.1,430.8 L131.6,430.1 L132.2,429.4
 L132.8,428.6 L133.3,427.9 L133.9,427.2 L134.5,426.5 L135.0,425.7 L135.6,425.0 L136.2,424.3 L136.8,423.6
 L137.3,422.9 L137.9,422.2 L138.5,421.5 L139.0,420.9 L139.6,420.2 L140.2,419.5 L140.7,418.8 L141.3,418.2
 L141.9,417.5 L142.4,416.8 L143.0,416.2 L143.6,415.5 L144.1,414.9 L144.7,414.2 L145.3,413.6 L145.8,412.9
 L146.4,412.3 L147.0,411.6 L147.5,411.0 L148.1,410.4 L148.7,409.8 L149.2,409.1 L149.8,408.5 L150.4,407.9
 L150.9,407.3 L151.5,406.7 L152.1,406.1 L152.6,405.4 L153.2,404.8 L153.8,404.2 L154.3,403.6 L154.9,403.0
 L155.5,402.5 L156.1,401.9 L156.6,401.3 L157.2,400.7 L157.8,400.1 L158.3,399.5 L158.9,398.9 L159.5,398.4
 L160.0,397.8 L160.6,397.2 L161.2,396.6 L161.7,396.1 L162.3,395.5 L162.9,394.9 L163.4,394.4 L164.0,393.8
 L164.6,393.3 L165.1,392.7 L165.7,392.2 L166.3,391.6 L166.8,391.1 L167.4,390.5 L168.0,390.0 L168.5,389.4
 L169.1,388.9 L169.7,388.3 L170.2,387.8 L170.8,387.3 L171.4,386.7 L171.9,386.2 L172.5,385.7 L173.1,385.1
 L173.6,384.6 L174.2,384.1 L174.8,383.6 L175.4,383.0 L175.9,382.5 L176.5,382.0 L177.1,381.5 L177.6,381.0
 L178.2,380.5 L178.8,379.9 L179.3,379.4 L179.9,378.9 L180.5,378.4 L181.0,377.9 L181.6,377.4 L182.2,376.9
 L182.7,376.4 L183.3,375.9 L183.9,375.4 L184.4,374.9 L185.0,374.4 L185.6,373.9 L186.1,373.4 L186.7,372.9
 L187.3,372.5 L187.8,372.0 L188.4,371.5 L189.0,371.0 L189.5,370.5 L190.1,370.0 L190.7,369.6 L191.2,369.1
 L191.8,368.6 L192.4,368.1 L192.9,367.6 L193.5,367.2 L194.1,366.7 L194.7,366.2 L195.2,365.8 L195.8,365.3
 L196.4,364.8 L196.9,364.4 L197.5,363.9 L198.1,363.4 L198.6,363.0 L199.2,362.5 L199.8,362.0 L200.3,361.6
 L200.9,361.1 L201.5,360.7 L202.0,360.2 L202.6,359.7 L203.2,359.3 L203.7,358.8 L204.3,358.4 L204.9,357.9
 L205.4,357.5 L206.0,357.0 L206.6,356.6 L207.1,356.1 L207.7,355.7 L208.3,355.3 L208.8,354.8 L209.4,354.4
 L210.0,353.9 L210.5,353.5 L211.1,353.1 L211.7,352.6 L212.2,352.2 L212.8,351.7 L213.4,351.3 L214.

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.2 
 
 
 
 
 0.3 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Interface position [m] 
 
 
 
 
 Time [s] 
 
 
 
 
 p=4 
 
 
 n=64 
 
 
 n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=32 
 
 
 n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=16 
 
 
 n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 n=8 
 
 
 n=8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 analytical Interface Position 
 
 
 analytical Interface Position 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M107.9,207.1 L161.3,207.1 M88.5,542.4 L89.1,529.5 L89.6,524.2 L90.2,520.1 L90.8,516.6 L91.3,513.6
 L91.9,510.8 L92.5,508.3 L93.0,506.0 L93.6,503.8 L94.2,501.7 L94.7,499.7 L95.3,497.8 L95.9,495.9
 L96.4,494.2 L97.0,492.5 L97.6,490.9 L98.2,489.3 L98.7,487.7 L99.3,486.2 L99.9,484.8 L100.4,483.4
 L101.0,482.0 L101.6,480.6 L102.1,479.3 L102.7,478.0 L103.3,476.7 L103.8,475.5 L104.4,474.2 L105.0,473.0
 L105.5,471.8 L106.1,470.7 L106.7,469.5 L107.2,468.4 L107.8,467.3 L108.4,466.2 L108.9,465.1 L109.5,464.0
 L110.1,463.0 L110.6,461.9 L111.2,460.9 L111.8,459.9 L112.3,458.9 L112.9,457.9 L113.5,456.9 L114.0,456.0
 L114.6,455.0 L115.2,454.1 L115.7,453.1 L116.3,452.2 L116.9,451.3 L117.5,450.4 L118.0,449.5 L118.6,448.6
 L119.2,447.7 L119.7,446.9 L120.3,446.0 L120.9,445.1 L121.4,444.3 L122.0,443.4 L122.6,442.6 L123.1,441.8
 L123.7,441.0 L124.3,440.1 L124.8,439.3 L125.4,438.5 L126.0,437.7 L126.5,436.9 L127.1,436.2 L127.7,435.4
 L128.2,434.6 L128.8,433.8 L129.4,433.1 L129.9,432.3 L130.5,431.6 L131.1,430.8 L131.6,430.1 L132.2,429.4
 L132.8,428.6 L133.3,427.9 L133.9,427.2 L134.5,426.5 L135.0,425.7 L135.6,425.0 L136.2,424.3 L136.8,423.6
 L137.3,422.9 L137.9,422.2 L138.5,421.5 L139.0,420.9 L139.6,420.2 L140.2,419.5 L140.7,418.8 L141.3,418.2
 L141.9,417.5 L142.4,416.8 L143.0,416.2 L143.6,415.5 L144.1,414.9 L144.7,414.2 L145.3,413.6 L145.8,412.9
 L146.4,412.3 L147.0,411.6 L147.5,411.0 L148.1,410.4 L148.7,409.8 L149.2,409.1 L149.8,408.5 L150.4,407.9
 L150.9,407.3 L151.5,406.7 L152.1,406.1 L152.6,405.4 L153.2,404.8 L153.8,404.2 L154.3,403.6 L154.9,403.0
 L155.5,402.5 L156.1,401.9 L156.6,401.3 L157.2,400.7 L157.8,400.1 L158.3,399.5 L158.9,398.9 L159.5,398.4
 L160.0,397.8 L160.6,397.2 L161.2,396.6 L161.7,396.1 L162.3,395.5 L162.9,394.9 L163.4,394.4 L164.0,393.8
 L164.6,393.3 L165.1,392.7 L165.7,392.2 L166.3,391.6 L166.8,391.1 L167.4,390.5 L168.0,390.0 L168.5,389.4
 L169.1,388.9 L169.7,388.3 L170.2,387.8 L170.8,387.3 L171.4,386.7 L171.9,386.2 L172.5,385.7 L173.1,385.1
 L173.6,384.6 L174.2,384.1 L174.8,383.6 L175.4,383.0 L175.9,382.5 L176.5,382.0 L177.1,381.5 L177.6,381.0
 L178.2,380.5 L178.8,379.9 L179.3,379.4 L179.9,378.9 L180.5,378.4 L181.0,377.9 L181.6,377.4 L182.2,376.9
 L182.7,376.4 L183.3,375.9 L183.9,375.4 L184.4,374.9 L185.0,374.4 L185.6,373.9 L186.1,373.4 L186.7,372.9
 L187.3,372.5 L187.8,372.0 L188.4,371.5 L189.0,371.0 L189.5,370.5 L190.1,370.0 L190.7,369.6 L191.2,369.1
 L191.8,368.6 L192.4,368.1 L192.9,367.6 L193.5,367.2 L194.1,366.7 L194.7,366.2 L195.2,365.8 L195.8,365.3
 L196.4,364.8 L196.9,364.4 L197.5,363.9 L198.1,363.4 L198.6,363.0 L199.2,362.5 L199.8,362.0 L200.3,361.6
 L200.9,361.1 L201.5,360.7 L202.0,360.2 L202.6,359.7 L203.2,359.3 L203.7,358.8 L204.3,358.4 L204.9,357.9
 L205.4,357.5 L206.0,357.0 L206.6,356.6 L207.1,356.1 L207.7,355.7 L208.3,355.3 L208.8,354.8 L209.4,354.4
 L210.0,353.9 L210.5,353.5 L211.1,353.1 L211.7,352.6 L212.2,352.2 L212.8,351.7 L213.4,351.3 L214.

Repeat the same for positional errors

In [19]:
foreach(int p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    Plot2Ddata PositionErrorAbs = new Plot2Ddata();
    Plot2Ddata PositionErrorRel = new Plot2Ddata();
    foreach(var dataGroup in dat.ElementAt(0).dataGroups){
        double[] times = dataGroup.Abscissas;
        double[] yValuesAbs = new double[times.Length];
        double[] yValuesRel = new double[times.Length];
        for(int i = 0; i < times.Length; i++){
            double y = dataGroup.Values[i];
            double y_ex = PhiFunc(times[i]);
            yValuesAbs[i] = y - y_ex;
            yValuesRel[i] = (y - y_ex)/y_ex;

        }
        Plot2Ddata errDataAbs = new Plot2Ddata(times, yValuesAbs, "abs-err-" + dataGroup.Name);
        PositionErrorAbs      = PositionErrorAbs.Merge(errDataAbs);
        Plot2Ddata errDataRel = new Plot2Ddata(times, yValuesRel, "rel-err-" + dataGroup.Name);
        PositionErrorRel      = PositionErrorRel.Merge(errDataRel);

    }

    // absolute errors
    var plot = PositionErrorAbs;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p1 = plot.PlotNow(); 
    
    plot = PositionErrorRel;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.002 
 
 
 
 
 -0.0018 
 
 
 
 
 -0.0016 
 
 
 
 
 -0.0014 
 
 
 
 
 -0.0012 
 
 
 
 
 -0.001 
 
 
 
 
 -0.0008 
 
 
 
 
 -0.0006 
 
 
 
 
 -0.0004 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 0.5 
 
 
 
 
 0.55 
 
 
 
 
 0.6 
 
 
 
 
 0.65 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=64 
 
 
 abs-err-n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=32 
 
 
 abs-err-n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=16 
 
 
 abs-err-n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=8 
 
 
 abs-err-n=8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

 <?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.35 
 
 
 
 
 -0.3 
 
 
 
 
 -0.25 
 
 
 
 
 -0.2 
 
 
 
 
 -0.15 
 
 
 
 
 -0.1 
 
 
 
 
 -0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 0.5 
 
 
 
 
 0.55 
 
 
 
 
 0.6 
 
 
 
 
 0.65 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=64 
 
 
 rel-err-n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=32 
 
 
 rel-err-n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=16 
 
 
 rel-err-n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=8 
 
 
 rel-err-n=8

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.0012 
 
 
 
 
 -0.0011 
 
 
 
 
 -0.001 
 
 
 
 
 -0.0009 
 
 
 
 
 -0.0008 
 
 
 
 
 -0.0007 
 
 
 
 
 -0.0006 
 
 
 
 
 -0.0005 
 
 
 
 
 -0.0004 
 
 
 
 
 -0.0003 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 0.5 
 
 
 
 
 0.55 
 
 
 
 
 0.6 
 
 
 
 
 0.65 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=64 
 
 
 abs-err-n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=32 
 
 
 abs-err-n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=16 
 
 
 abs-err-n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=8 
 
 
 abs-err-n=8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

 <?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.3 
 
 
 
 
 -0.25 
 
 
 
 
 -0.2 
 
 
 
 
 -0.15 
 
 
 
 
 -0.1 
 
 
 
 
 -0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 0.5 
 
 
 
 
 0.55 
 
 
 
 
 0.6 
 
 
 
 
 0.65 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=64 
 
 
 rel-err-n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=32 
 
 
 rel-err-n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=16 
 
 
 rel-err-n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=8 
 
 
 rel-err-n=8

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.001 
 
 
 
 
 -0.0009 
 
 
 
 
 -0.0008 
 
 
 
 
 -0.0007 
 
 
 
 
 -0.0006 
 
 
 
 
 -0.0005 
 
 
 
 
 -0.0004 
 
 
 
 
 -0.0003 
 
 
 
 
 -0.0002 
 
 
 
 
 -0.0001 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 0.5 
 
 
 
 
 0.55 
 
 
 
 
 0.6 
 
 
 
 
 0.65 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=64 
 
 
 abs-err-n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=32 
 
 
 abs-err-n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=16 
 
 
 abs-err-n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=8 
 
 
 abs-err-n=8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

 <?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.3 
 
 
 
 
 -0.25 
 
 
 
 
 -0.2 
 
 
 
 
 -0.15 
 
 
 
 
 -0.1 
 
 
 
 
 -0.05 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 0.5 
 
 
 
 
 0.55 
 
 
 
 
 0.6 
 
 
 
 
 0.65 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=64 
 
 
 rel-err-n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=32 
 
 
 rel-err-n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=16 
 
 
 rel-err-n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=8 
 
 
 rel-err-n=8

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.001 
 
 
 
 
 -0.0009 
 
 
 
 
 -0.0008 
 
 
 
 
 -0.0007 
 
 
 
 
 -0.0006 
 
 
 
 
 -0.0005 
 
 
 
 
 -0.0004 
 
 
 
 
 -0.0003 
 
 
 
 
 -0.0002 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 0.5 
 
 
 
 
 0.55 
 
 
 
 
 0.6 
 
 
 
 
 0.65 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=64 
 
 
 abs-err-n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=32 
 
 
 abs-err-n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=16 
 
 
 abs-err-n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-n=8 
 
 
 abs-err-n=8 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

 <?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.3 
 
 
 
 
 -0.25 
 
 
 
 
 -0.2 
 
 
 
 
 -0.15 
 
 
 
 
 -0.1 
 
 
 
 
 -0.05 
 
 
 
 
 0 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0.35 
 
 
 
 
 0.4 
 
 
 
 
 0.45 
 
 
 
 
 0.5 
 
 
 
 
 0.55 
 
 
 
 
 0.6 
 
 
 
 
 0.65 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=64 
 
 
 rel-err-n=64 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=32 
 
 
 rel-err-n=32 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=16 
 
 
 rel-err-n=16 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rel-err-n=8 
 
 
 rel-err-n=8

### Spatial Convergence

Evaluated at last timestep  
Plot Positional error over hmin

Hint use v1

In [20]:
var p = new Plot2Ddata();
foreach(var (k, group) in groupedSessions){
    int i = 0;

    List<double> XValues = new List<double>();
    List<double> YValues = new List<double>();

    double t_End = data[k].ElementAt(0).dataGroups.First().Abscissas.Last();
    double x_Exact = PhiFuncF.Evaluate(new double[2], t_End);

    foreach(var s in group){
        display(s.Name);        
        var h = Convert.ToDouble(s.KeysAndQueries["Grid:hMin"]);
        var x = data[k].ElementAt(0).dataGroups[i].Values.Last();
        double err = Math.Abs(x - x_Exact);
        XValues.Add(h);
        YValues.Add(err);
        i++;
    }
    p = p.Merge(new Plot2Ddata(XValues, YValues, "P = " + k));    
}

p.ModFormat();
//p.AddDataGroup(" O(1) ", new[] {1e-4, 1e-2}, new[] {1e-4, 1e-2}, "r-");
p.AddDataGroup(" O(2) ", new[] {1e-4, 1e-2}, new[] {1e-5, 1e-1}, "r-");
//p.AddDataGroup(" O(3) ", new[] {1e-4, 1e-3}, new[] {1e-5, 1e-2}, "r--");
p.LogX = true;
p.LogY = true;
var gp = p.ToGnuplot();
gp.WriteScriptAndData("./Figures/gnuplot/data/Suckingproblem_SpatialConvergence"); // change a few settings by hand, see readme!

HtmlString pp = new HtmlString(
    "<div>" +  
    "   <div>" +
        p.PlotNow().ToString() +
    "   </div>" +
    "</div>");
display(pp);

XNSFE-SuckingProblem-v1_H64_AMR0_P1

XNSFE-SuckingProblem-v1_H32_AMR0_P1

XNSFE-SuckingProblem-v1_H16_AMR0_P1

XNSFE-SuckingProblem-v1_H8_AMR0_P1

XNSFE-SuckingProblem-v1_H64_AMR0_P2

...

XNSFE-SuckingProblem-v1_H32_AMR0_P4

XNSFE-SuckingProblem-v1_H16_AMR0_P4

XNSFE-SuckingProblem-v1_H8_AMR0_P4

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 $10 -5 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 $10 -1 $ 
 
 
 
 
 $10 0 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 $10 -1 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 P = 1 
 
 
 P = 1 
 
 
 
 
 
 
 
 
 
 
 P = 2 
 
 
 P = 2 
 
 
 
 
 
 
 
 
 
 
 P = 3 
 
 
 P = 3 
 
 
 
 
 
 
 
 
 
 
 P = 4 
 
 
 P = 4 
 
 
 
 
 
 
 
 
 
 
 O(2) 
 
 
 O(2)

### Temporal Convergence

Hint use v3 : errors are compared after 50 time steps on the coarsest dt

In [21]:
string proj = "XNSFE-SuckingProblem-v3";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

Project name is set to 'XNSFE-SuckingProblem-v3'.
Opening existing database '\\dc1\userspace\rieckmann\cluster\XNSFE-SuckingProblem-v3'.


In [22]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination && Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "T");
//sessions = sessions.Where(s => s.Name.Contains("Splitting")).ToList();
//sessions = sessions.Where(s => s.Name.Contains("SDIRK") || s.Name.Contains("Euler") ).ToList();
sessions = sessions.Where(s => s.Name.Contains("LieSplitting")).ToList();
sessions = sessions.Where(s => s.Name.Contains("ImplicitEuler") && !s.Name.Contains("RK")).ToList();
sessions = sessions.OrderBy(s => s.KeysAndQueries["dtFixed"]).ToList();
sessions.ForEach(s => display(s))

XNSFE-SuckingProblem-v3	XNSFE-SuckingProblem-v3_H64_AMR0_P4_T3_LieSplitting_ImplicitEuler	11/30/2022 15:46:10	8be0c424...

XNSFE-SuckingProblem-v3	XNSFE-SuckingProblem-v3_H64_AMR0_P4_T2_LieSplitting_ImplicitEuler	11/30/2022 15:44:49	db8b1de1...

XNSFE-SuckingProblem-v3	XNSFE-SuckingProblem-v3_H64_AMR0_P4_T1_LieSplitting_ImplicitEuler	11/30/2022 15:43:27	36383e16...

XNSFE-SuckingProblem-v3	XNSFE-SuckingProblem-v3_H64_AMR0_P4_T0_LieSplitting_ImplicitEuler	11/30/2022 15:42:08	0ac89af0...

In [23]:
//Dictionary<int, List<ISessionInfo>> groupedSessions = sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["Timestepper_LevelSetHandling"])).ToDictionary(g => g.Key, g => g.ToList());
Dictionary<Tuple<int, string>, List<ISessionInfo>> groupedSessions = sessions.GroupBy(s => new Tuple<int, string>(Convert.ToInt32(s.KeysAndQueries["Timestepper_LevelSetHandling"]), Convert.ToString((BoSSS.Solution.XdgTimestepping.TimeSteppingScheme)Convert.ToInt32(s.KeysAndQueries["TimeSteppingScheme"])))).ToDictionary(g => g.Key, g => g.ToList());
groupedSessions.Keys

index,value
0,"(1, ImplicitEuler)"


In [24]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("StefanProblem"));

Element at 0: time vs interface-x-pos-min
Element at 1: time vs interface-x-pos-max
Element at 2: time vs mass-vapor
Element at 3: time vs mass-liquid
Element at 4: time vs massflux-interface
Element at 5: time vs massflux-outlet


In [25]:
data.First().Value.ElementAt(0).dataGroups[0].Abscissas.Last()

0.0070243755972244455

In [26]:
var p = new Plot2Ddata();
foreach(var (k, group) in groupedSessions){
    int i = 0;

    List<double> XValues = new List<double>();
    List<double> YValues = new List<double>();

    double t_End = data[k].ElementAt(0).dataGroups[i].Abscissas.Last();
    double x_Exact = PhiFuncF.Evaluate(new double[2], t_End);
    
    foreach(var s in group){
        display(s.Name);        
        var h = Convert.ToDouble(s.KeysAndQueries["dtFixed"]);
        var x = data[k].ElementAt(0).dataGroups[i].Values.Last();
        double err = Math.Abs(x - x_Exact);
        XValues.Add(h);
        YValues.Add(err);
        i++;
    }
    p = p.Merge(new Plot2Ddata(XValues, YValues, "Splitting Order = " + k.Item1 + " Scheme : " + k.Item2));    
}

p.LegendAlignment = new[] {"o","c","r"};
p.ModFormat();
p.AddDataGroup(" O(1) ", new[] {1e-5, 1e-3}, new[] {1e-8, 1e-6}, "r-");
//p.AddDataGroup(" O(2) ", new[] {2e-5, 1e-4}, new[] {1e-8, 2.5e-7}, "r-");
var gp1 = p.ToGnuplot();
gp1.WriteScriptAndData("./Figures/gnuplot/data/Suckingproblem_TemporalConvergence"); // change a few settings by hand, see readme!

p.LogX = true;
p.LogY = true;
p.XrangeMin = 1 * 1e-5;
p.XrangeMax = 2 * 1e-4;
Gnuplot gp = new Gnuplot();
p.ToGnuplot(gp);
HtmlString pp = new HtmlString(
    "<div>" +  
    "   <div>" +
        gp.PlotSVG(xRes: 1600).ToString() +
    "   </div>" +
    "</div>");
display(pp);

XNSFE-SuckingProblem-v3_H64_AMR0_P4_T3_LieSplitting_ImplicitEuler

XNSFE-SuckingProblem-v3_H64_AMR0_P4_T2_LieSplitting_ImplicitEuler

XNSFE-SuckingProblem-v3_H64_AMR0_P4_T1_LieSplitting_ImplicitEuler

XNSFE-SuckingProblem-v3_H64_AMR0_P4_T0_LieSplitting_ImplicitEuler

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 $10 -8 $ 
 
 
 
 
 $10 -7 $ 
 
 
 
 
 $10 -6 $ 
 
 
 
 
 $10 -5 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Splitting Order = 1 Scheme : ImplicitEuler 
 
 
 Splitting Order = 1 Scheme : ImplicitEuler 
 
 
 
 
 
 
 
 
 
 
 O(1) 
 
 
 O(1)

In [27]:
sessions.ForEach(s => display(s.KeysAndQueries["dtFixed"]));

1.756093899306124E-05

3.512187798612248E-05

7.024375597224497E-05

0.00014048751194448993

In [28]:
foreach(var p in data.Keys.OrderBy(x => x)){
    var dat = data[p];
    Plot2Ddata PositionErrorAbs = new Plot2Ddata();
    Plot2Ddata PositionErrorRel = new Plot2Ddata();
    foreach(var dataGroup in dat.ElementAt(0).dataGroups){
        double[] times = dataGroup.Abscissas;
        double[] yValuesAbs = new double[times.Length];
        double[] yValuesRel = new double[times.Length];
        for(int i = 0; i < times.Length; i++){
            double y = dataGroup.Values[i];
            double y_ex = PhiFunc(times[i]);
            yValuesAbs[i] = y - y_ex;
            yValuesRel[i] = (y - y_ex)/y_ex;

        }
        Plot2Ddata errDataAbs = new Plot2Ddata(times, yValuesAbs, "abs-err-" + dataGroup.Name);
        PositionErrorAbs      = PositionErrorAbs.Merge(errDataAbs);
        Plot2Ddata errDataRel = new Plot2Ddata(times, yValuesRel, "rel-err-" + dataGroup.Name);
        PositionErrorRel      = PositionErrorRel.Merge(errDataRel);

    }

    // absolute errors
    var plot = PositionErrorAbs;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p1 = plot.PlotNow(); 
    
    plot = PositionErrorRel;
    plot.ModFormat();
    plot.LegendAlignment = new string[] {"o", "t", "c"}; 
    var p2 = plot.PlotNow();

    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            p1.ToString() +
        "   </div>" +
        "   <div>" +
            p2.ToString() +
        "   </div style='float:right;'>" +
        "</div>");
    display(pp);
}

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -2x10 -8 
 
 
 
 
 0 
 
 
 
 
 2x10 -8 
 
 
 
 
 4x10 -8 
 
 
 
 
 6x10 -8 
 
 
 
 
 8x10 -8 
 
 
 
 
 1x10 -7 
 
 
 
 
 1.2x10 -7 
 
 
 
 
 0 
 
 
 
 
 0.001 
 
 
 
 
 0.002 
 
 
 
 
 0.003 
 
 
 
 
 0.004 
 
 
 
 
 0.005 
 
 
 
 
 0.006 
 
 
 
 
 0.007 
 
 
 
 
 0.008 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 abs-err-XNSFE-SuckingProblem-v3-H64-AMR0-P4-T3-LieSplitting-ImplicitEuler 
 
 
 abs-err-XNSFE-SuckingProblem-v3-H64-AMR0-P4-T3-LieSplitting-ImplicitEuler 
 
 
 
	<path stroke='rgb( 0, 0, 0)' d='M0.0,21.0 L40.2,21.0 M87.1,502.3 L88.6,502.3 L90.0,502.0 L91.5,501.7 L92.9,501.5 L94.4,501.2
 L95.8,501.0 L97.3,500.7 L98.7,500.5 L100.2,500.3 L101.7,500.0 L103.1,499.8 L104.6,499.6 L106.0,499.4
 L107.5,499.2 L108.9,499.0 L110.4,498.8 L111.8,498.6 L113.3,498.4 L114.8,498.2 L116.2,498.0 L117.7,497.8
 L119.1,497.6 L120.6,497.4 L122.0,497.3 L123.5,497.1 L124.9,496.9 L126.4,496.8 L127.9,496.6 L129.3,496.4
 L130.8,496.3 L132.2,496.1 L133.7,495.9 L135.1,495.8 L136.6,495.6 L138.0,495.5 L139.5,495.3 L140.9,495.2
 L142.4,495.1 L143.9,494.9 L145.3,494.8 L146.8,494.6 L148.2,494.5 L149.7,494.4 L151.1,494.2 L152.6,494.1
 L154.0,494.0 L155.5,493.8 L157.0,493.7 L158.4,493.6 L159.9,493.4 L161.3,493.3 L162.8,493.2 L164.2,493.1
 L165.7,493.0 L167.1,492.8 L168.6,492.7 L170.1,492.6 L171.5,492.5 L173.0,492.4 L174.4,492.3 L175.9,492.1
 L177.3,492.0 L178.8,491.9 L180.2,491.8 L181.7,491.7 L183.2,491.6 L184.6,491.5 L186.1,491.4 L187.5,491.3
 L189.0,491.2 L190.4,491.0 L191.9,490.9 L193.3,490.8 L194.8,490.7 L196.3,490.6 L197.7,490.5 L199.2,490.4
 L200.6,490.3 L202.1,490.2 L203.5,490.1 L205.0,490.0 L206.4,489.9 L207.9,489.8 L209.4,489.7 L210.8,489.6
 L212.3,489.5 L213.7,489.4 L215.2,489.3 L216.6,489.2 L218.1,489.1 L219.5,489.0 L221.0,488.9 L222.4,488.8
 L223.9,488.7 L225.4,488.6 L226.8,488.5 L228.3,488.4 L229.7,488.3 L231.2,488.2 L232.6,488.1 L234.1,488.0
 L235.5,487.9 L237.0,487.8 L238.5,487.7 L239.9,487.6 L241.4,487.5 L242.8,487.5 L244.3,487.4 L245.7,487.3
 L247.2,487.2 L248.6,487.1 L250.1,487.0 L251.6,486.9 L253.0,486.8 L254.5,486.7 L255.9,486.6 L257.4,486.5
 L258.8,486.4 L260.3,486.3 L261.7,486.2 L263.2,486.1 L264.7,486.0 L266.1,485.9 L267.6,485.8 L269.0,485.7
 L270.5,485.6 L271.9,485.5 L273.4,485.4 L274.8,485.3 L276.3,485.2 L277.8,485.1 L279.2,485.0 L280.7,484.9
 L282.1,484.8 L283.6,484.7 L285.0,484.6 L286.5,484.5 L287.9,484.4 L289.4,484.3 L290.9,484.2 L292.3,484.1
 L293.8,484.0 L295.2,483.9 L296.7,483.8 L298.1,483.7 L299.6,483.6 L301.0,483.5 L302.5,483.4 L303.9,483.3
 L305.4,483.2 L306.9,483.1 L308.3,482.9 L309.8,482.8 L311.2,482.7 L312.7,482.6 L314.1,482.5 L315.6,482.4
 L317.0,482.3 L318.5,482.2 L320.0,482.1 L321.4,482.0 L322.9,481.9 L324.3,481.8 L325.8,481.7 L327.2,481.6
 L328.7,481.5 L330.1,481.3 L331.6,481.2 L333.1,481.1 L334.5,481.0 L336.0,480.9 L337.4,480.8 L338.9,480.7
 L340.3,480.6 L341.8,480.5 L343.2,480.3 L344.7,480.2 L346.2,480.1 L347.6,480.0 L349.1,479.9 L350.5,479.8
 L352.0,479.7 L353.4,479.6 L354.9,479.4 L356.3,479.3 L357.8,479.2 L359.3,479.1 L360.7,479.0 L362.2,478.9
 L363.6,478.7 L365.1,478.6 L366.5,478.5 L368.0,478.4 L369.4,478.3 L370.9,478.2 L372.4,478.0 L373.8,477.9
 L375.3,477.8 L376.7,477.7 L378.2,477.6 L379.6,477.4 L381.1,477.3 L382.5,477.2 L384.0,477.1 L385.4,477.0
 L386.9,476.8 L388.4,476.7 L389.8,476.6 L391.3,476.5 L392.7,476.4 L394.2,476.2 L395.6,476.1 L397.1,476.0
 L398.5,475.9 L400.0,475.7 L401.5,475.6 L402.9,475.5 L404.4,475.4 L405.8,475.2 L407.3,475.1 L408.7,475.0
 L410.2,474.9 L411.6,474.7 L413.1,474.6 L414.6,474.5 L416.0,474.4 L417.5,474.2 L418.9,474.1 L420.4,474.0
 L421.8,473.8 L423.3,473.7 L424.7,473.6 L4

### L2 Convergence

We tricks a little and use the exact solution to compare against not as a function of time, but of interface position!  
Use v1 again.

In [29]:
BoSSSshell.WorkflowMgm.Init("XNSFE-SuckingProblem-v1");

Project name is set to 'XNSFE-SuckingProblem-v1'.


In [41]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination && Convert.ToString(s.KeysAndQueries["id:Convtype"]) == "H");
sessions = sessions.OrderBy(s => s.KeysAndQueries["Grid:hMax"]).ToList();
sessions

#0: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P4	11/14/2022 13:20:50	1ea7f318...
#1: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P2	11/14/2022 13:20:31	609eb176...
#2: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P3	11/14/2022 13:20:41	9d51c326...
#3: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H64_AMR0_P1	11/14/2022 13:20:21	57ebe3d7...
#4: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H32_AMR0_P4	11/14/2022 13:20:12	82541579...
#5: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H32_AMR0_P3	11/14/2022 13:20:02	760d3d43...
#6: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H32_AMR0_P1	11/14/2022 13:19:43	dcd82dde...
#7: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H32_AMR0_P2	11/14/2022 13:19:53	834c1c20...
#8: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H16_AMR0_P3	11/14/2022 13:19:25	41975e70...
#9: XNSFE-SuckingProblem-v1	XNSFE-SuckingProblem-v1_H16_AMR0_P4	11/14/2022 13:19:35	93b4ca77...
#10: XNSFE-SuckingProblem-v1	XNSFE-Sucki

In [42]:
SortedDictionary<int, List<ISessionInfo>> groupedSessions = new SortedDictionary<int, List<ISessionInfo>>(sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"])).ToDictionary(g => g.Key, g => g.ToList()));

In [32]:
// var sess = sessions.Pick(49);
// sess
//sess.Timesteps.Last().Export().WithShadowFields().To(Path.GetFullPath("./Plot/"+sess.Name)).WithSupersampling(2).Do()

#### Utility Functions

In [33]:
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Quadrature;

In [34]:
// Code functions of V_x, T, p as Functions of f(x, t(xI)) by inversing the relationship xI(t)
public static double InterfaceTime(double xI){
    return Math.Pow(xI / (2 * Constants.Xi),2) /  Constants.alpha_B - Constants.t0;
}

public static double InterfaceVel(double[] X, double t){
    return Constants.Xi * Constants.alpha_B / Math.Sqrt(Constants.alpha_B * (t+Constants.t0));
}

public static Func<double[], double>[] GetV(double xI){
    double tI = InterfaceTime(xI);
    return new Func<double[], double>[] { (X) => Constants.eps * InterfaceVel(X,tI), (X) => 0.0};
}

public static Func<double[], double>[] GetT(double xI){
    string path = @".\SuckingProblemRef.csv";
    string[] lines = File.ReadAllLines(path);
    double[] eta = new double[lines.Length];
    double[][] valueDat = new double[2][];
    for(int j = 0; j < 2; j++)
        valueDat[j] = new double[lines.Length];

    for (int i = 0; i < lines.Length; i++) {
        eta[i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[0]);            
        for (int j = 0; j < 2; j++) {
            valueDat[j][i] = Convert.ToDouble(lines[i].Split(new string[] { "," }, StringSplitOptions.RemoveEmptyEntries)[j+1]);
            if(j == 0) valueDat[j][i] = (valueDat[j][i] - 373.15) / 5.0;
        }
    } 
    
    double _xI = xI;
    double _tI = InterfaceTime(xI);

    double[] x = new double[lines.Length+2];
    double[] y = new double[lines.Length+2];
    x[0] = 0.0;
    y[0] = Constants.T_sat-_xI * Constants.Xi / Constants.C * Math.Sqrt(Constants.alpha_B / (Constants.t0 + _tI)) ; // linear extrapolation node, using slope of temperature at interface
    for(int i = 1; i < lines.Length + 1; i++){
        y[i] = valueDat[0][i-1];
        double xi = eta[i-1]* Math.Sqrt(2 * Constants.alpha_A * (Constants.t0 + _tI));
        x[i] = _xI + xi;
    }
    x[lines.Length+1] = Constants.L;
    y[lines.Length+1] = Constants.T_wall;
    Spline1D TempAInitial = new Spline1D(x,y,0);

    Func<double[], double> TA = (X) => {        
        return TempAInitial.Evaluate(X,0.0);
    };

    return new Func<double[], double>[] { TA, (X) => 0.0};
}

public static double GetXI(ITimestepInfo ts){
    XDGField field = (XDGField)ts.Fields.Where(f => f is XDGField).First();
    var LsTrk = field.Basis.Tracker;
    var helper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS, 0).XQuadSchemeHelper;
    var rule = helper.GetLevelSetquadScheme(0, LsTrk.Regions.GetCutCellMask()).Compile(LsTrk.GridDat, 0);

    double x = 0.0;
    foreach(var part in rule){
        foreach(int i in part.Chunk.Elements){
            MultidimensionalArray glob = MultidimensionalArray.Create(part.Rule.Nodes.Lengths);
            LsTrk.GridDat.TransformLocal2Global(part.Rule.Nodes, glob, i);
            x = Math.Max(x, glob.ExtractSubArrayShallow(-1, 0).Max());
        }
    }

    //Console.WriteLine("ts : {0}, x : {1}", ts.Grid.NumberOfCells, x);
    return x;
}

In [35]:
public static double ComputeErrorTemperature(ITimestepInfo ts, int mode = 0){    
    double xI = GetXI(ts);
    var exact = GetT(xI);
    var field = ts.Fields.Single(f => f.Identification == "Temperature").As<XDGField>();
    double err = ComputeErrors(field, exact, mode);
    return err;
}

public static double ComputeErrorVelocity(ITimestepInfo ts, int mode = 0){
    double xI = GetXI(ts);
    var exact = GetV(xI);
    //Console.WriteLine("Expected velocity {0}", exact[0](new[] {0.0, 0.0}));
    var field = ts.Fields.Single(f => f.Identification == "VelocityX").As<XDGField>();
    double err = ComputeErrors(field, exact, mode);
    //Console.WriteLine("Velocity error : {0}", err);
    return err;
}

public static double ComputeErrors(XDGField field, Func<double[], double>[] ExactSolution, int mode = 0){
    int order = 12;    
    var LsTrk = field.Basis.Tracker;
    int N = field.GridDat.iGeomCells.NoOfLocalUpdatedCells;
    double B = Constants.L / N;
    double b = Math.Sqrt(B);
    //Console.WriteLine(LsTrk.CutCellQuadratureType);
    var schemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS, order).XQuadSchemeHelper;
    XDGField Solution = new XDGField(new XDGBasis(LsTrk, 6));
    Solution.GetSpeciesShadowField("A").ProjectField(ExactSolution[0]);
    Solution.GetSpeciesShadowField("B").ProjectField(ExactSolution[1]);

    var schemeA = schemeHelper.GetVolumeQuadScheme(LsTrk.SpeciesIdS[0]);

    double errA = 0.0;
    double areaA = 0.0;
    CellQuadrature.GetQuadrature(new[] { 2 }, LsTrk.GridDat, schemeA.Compile(LsTrk.GridDat, order), 
    delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
        Solution.GetSpeciesShadowField("A").Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1, -1, 0), 1.0 );
        field.GetSpeciesShadowField("A").Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1, -1, 0), -1.0 );
        EvalResult.ExtractSubArrayShallow(-1, -1, 0).ApplyAll(x => x*x);
        EvalResult.ExtractSubArrayShallow(-1, -1, 1).SetAll(1.0);
    },
    delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
        for (int i = 0; i < Length; i++){
            errA += ResultsOfIntegration[i, 0];
            areaA += ResultsOfIntegration[i, 1];
        }
    }).Execute();
    errA = Math.Sqrt(errA);

    var schemeB = schemeHelper.GetVolumeQuadScheme(LsTrk.SpeciesIdS[1]);

    double errB = 0.0;
    double areaB = 0.0;
    CellQuadrature.GetQuadrature(new[] { 2 }, LsTrk.GridDat, schemeB.Compile(LsTrk.GridDat, order), 
    delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
        Solution.GetSpeciesShadowField("B").Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1, -1, 0), 1.0 );
        field.GetSpeciesShadowField("B").Evaluate(i0, Length, QR.Nodes, EvalResult.ExtractSubArrayShallow(-1, -1, 0), -1.0 );
        EvalResult.ExtractSubArrayShallow(-1, -1, 0).ApplyAll(x => x*x);
        EvalResult.ExtractSubArrayShallow(-1, -1, 1).SetAll(1.0);

    },
    delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
         for (int i = 0; i < Length; i++){
            errB += ResultsOfIntegration[i, 0];
            areaB += ResultsOfIntegration[i, 1];
        }
    }).Execute();
    errB = Math.Sqrt(errB);

    switch(mode){
        case 0:
        default:
            return Math.Sqrt(errA * errA + errB * errB) / b;
        case 1:
            return errB / b;
        case -1:
            return errA / b;
        case 2:
            return errB;
        case -2:
            return errA;
        case 3:
            return areaB;
        case -3:
            return areaA;
        case 4:
            return errB / Math.Sqrt(areaB);
        case -4:
            return errA / Math.Sqrt(areaA);
        case 10:
            return areaA + areaB;
        case 11:
            return Math.Sqrt(errA * errA + errB * errB);
    }
}

In [36]:
public static Plot2Ddata L2Errors(IEnumerable<ISessionInfo> sessions){
    Plot2Ddata plot = new Plot2Ddata();
    List<double> errT = new List<double>();
    List<double> errV = new List<double>();
    List<double> h = new List<double>();

    foreach(var s in sessions){
        Console.WriteLine(s.Name);
        errT.Add(ComputeErrorTemperature(s.Timesteps.Last()));
        errV.Add(ComputeErrorVelocity(s.Timesteps.Last()));
        h.Add(Constants.L/s.Timesteps.Last().Grid.NumberOfCells);
    }

    plot.AddDataGroup("Temperature Error", h, errT, "k-x");
    plot.AddDataGroup("Velocity Error", h, errV, "k-+");
    plot.AddDataGroup(" O(1) ", new[] {h.Min(), h.Max()}, new[] {errV.Min(), h.Max()/h.Min()*errV.Min()}, "r-");

    plot.Xlabel = "cell size [m]";
    plot.Ylabel = "normalized error";
    plot.LogX = true;
    plot.LogY = true;

    return plot;
}

public static Plot2Ddata[] L2Errors(SortedDictionary<int, List<ISessionInfo>> groupedSessions, int mode = 0){
    Plot2Ddata plotT = new Plot2Ddata();
    Plot2Ddata plotV = new Plot2Ddata();

    foreach(var kvp in groupedSessions){
        List<double> errT = new List<double>();
        List<double> errV = new List<double>();
        List<double> h = new List<double>();

        foreach(var s in kvp.Value){
            var ts = s.Timesteps.Last();
            Console.WriteLine(s.Name + " t = {0}", ts.PhysicalTime);
            errT.Add(ComputeErrorTemperature(ts, mode));
            errV.Add(ComputeErrorVelocity(ts, mode));
            h.Add(Constants.L/ts.Grid.NumberOfCells);
        }

        plotT.AddDataGroup("p = " + kvp.Key, h, errT, "k-x");
        plotV.AddDataGroup("p = " + kvp.Key, h, errV, "k-x");
    }

    //plot.AddDataGroup(" O(1) ", new[] {h.Min(), h.Max()}, new[] {errV.Min(), h.Max()/h.Min()*errV.Min()}, "r-");
    plotT.ModFormat();
    plotT.Xlabel = "cell size [m]";
    plotT.Ylabel = "temperature error";
    plotT.LogX = true;
    plotT.LogY = true;

    plotV.ModFormat();
    plotV.Xlabel = "cell size [m]";
    plotV.Ylabel = "velocity error";
    plotV.LogX = true;
    plotV.LogY = true;

    return new[] {plotT, plotV};
}

#### Evaluation

In [43]:
var plt = L2Errors(groupedSessions, -4);

XNSFE-SuckingProblem-v1_H64_AMR0_P1 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H32_AMR0_P1 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H16_AMR0_P1 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H8_AMR0_P1 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H64_AMR0_P2 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H32_AMR0_P2 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H16_AMR0_P2 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H8_AMR0_P2 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H64_AMR0_P3 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H32_AMR0_P3 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H16_AMR0_P3 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H8_AMR0_P3 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H64_AMR0_P4 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H32_AMR0_P4 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H16_AMR0_P4 t = 0.49957359247463495
XNSFE-SuckingProblem-v1_H8_AMR0_P4 t = 0.49957359247463495


In [44]:
plt[0].PlotNow()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 $10 -1 $ 
 
 
 
 
 $10 0 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 temperature error 
 
 
 
 
 cell size [m] 
 
 
 
 
 p = 1 
 
 
 p = 1 
 
 
 
 
 
 
 
 
 
 
 p = 2 
 
 
 p = 2 
 
 
 
 
 
 
 
 
 
 
 p = 3 
 
 
 p = 3 
 
 
 
 
 
 
 
 
 
 
 p = 4 
 
 
 p = 4

In [45]:
plt[1].PlotNow()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 $10 -6 $ 
 
 
 
 
 $10 -5 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 velocity error 
 
 
 
 
 cell size [m] 
 
 
 
 
 p = 1 
 
 
 p = 1 
 
 
 
 
 
 
 
 
 
 
 p = 2 
 
 
 p = 2 
 
 
 
 
 
 
 
 
 
 
 p = 3 
 
 
 p = 3 
 
 
 
 
 
 
 
 
 
 
 p = 4 
 
 
 p = 4

In [46]:
GetV(0.00221)[0](new[] {0.0, 0.0})

0.010995622739815518

### Animation of Temperature and Velocity

In [47]:
var sess2plot = groupedSessions[3].OrderBy(s => s.KeysAndQueries["Grid:hMax"]).First();// pick session with finest grid and highest degree
string path2plot = Path.GetFullPath(Path.Combine("./Plots", sess2plot.Name));
sess2plot.Export().To(path2plot).WithSupersampling(2).WithShadowFields().Do(true);

Starting export process... Data will be written to the directory: d:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-Master\internal\src\private-rckmnn\notebooks\XNSFE-Paper\1D-Sucking\Plots\XNSFE-SuckingProblem-v1_H64_AMR0_P3


In [51]:
#!pwsh
#!share --from csharp path2plot
pvpython ./Plots/suckingproblem.py $path2plot

loading plt from  d:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-Master\internal\src\private-rckmnn\notebooks\XNSFE-Paper\1D-Sucking\Plots\XNSFE-SuckingProblem-v1_H64_AMR0_P3


In [ ]:
display(new HtmlString(String.Format("<video width=\"1487\" height=\"1121\" controls><source src=\"{0}\" type=\'video/ogg; codecs=\"theora\"\'></video>",Path.Combine("./Plots", sess2plot.Name, "suckingproblem.ogv") )))

Add more postprocessing, look up what you have already done last year Matthias!  
Especially how to obtain some meaningful EOC?